# Tutorial 1 using Unit Commitment Example

This tutorial shows the different commands for using GEMSPy python package for modeling and simulating an Unit Commitment problem.

We use in this tutorial the library, *antares-legacy-models*, present in GEMS repo.

## Installation of GEMSPy

In [1]:
pip install gemspy

  Using cached gemspy-0.0.4-py3-none-any.whl.metadata (23 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached ortools-9.15.6755-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.3 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached antlr4_python3_runtime-4.13.2-py3-none-any.whl.metadata (304 bytes)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached antares_craft-0.11.0-py3-none-any.whl.metadata (25 kB)
  Using cached anytree-2.13.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached antares_timeseries_generation-0.1.9-py3-none-any.whl.metadata (2.1 kB)
  Using cached antares_study_version-1.0.20-py3-none-any.whl.metadata (2.4 kB)
  Using cached pandas-2.3.3-cp312-cp312-ma

## Create a simple system composed of an area with load

### Step 1: Configure an Area

An area is the central node in the electrical grid where different components connect. We'll create an area with spillage cost and unsupplied energy cost parameters.

In [ ]:
from gems.study.parsing import InputComponent, InputComponentParameter, InputSystem
from gems.model.parsing import parse_yaml_library
from gems.model.resolve_library import resolve_library
from pathlib import Path

# Set tutorial directory structure
study_name = "QSE_2_Unit_Commitment_tuto"

base_path = Path.cwd()
library_folder = base_path / study_name / "input" / "model-libraries"
library_file = next(library_folder.glob("*.yml")).name

series_folder = base_path / study_name / "input" / "data-series"

# Read library file containing the models
print("LIBRARY READING")
print("\tLibrary path:", library_file)

with open(Path(library_folder, library_file)) as lib_file:
    input_libraries = [parse_yaml_library(lib_file)]
    library_name = input_libraries[0].id
    print("\tLibrary loaded:", library_name)

result_lib = resolve_library(input_libraries)

print("COMPONENTS DEFINITION")
components = []

# Add an area component with spillage and ENS costs
components.append(
    InputComponent(
        id="my_area",
        model=f"{library_name}.area",
        parameters=[
            InputComponentParameter(
                id="spillage_cost",
                time_dependent=False,
                scenario_dependent=False,
                value=1000),
            InputComponentParameter(
                id="ens_cost",
                time_dependent=False,
                scenario_dependent=False,
                value=10000),
        ],
    )
)

# Add a load component with load data from a data series
components.append(
    InputComponent(
        id="load",
        model=f"{library_name}.load",
        parameters=[
            InputComponentParameter(
                id="load",
                time_dependent=True,
                scenario_dependent=True,
                value="load"),
        ],
    )
)
print("\tComponents defined:", [comp.id for comp in components])

print("INPUT SYSTEM CREATION")
input_system = InputSystem(
            id = "my_system",
            model_libraries=library_name,
            components=components,
        )
print("\tThe following input system has been created :")
print(input_system)

LIBRARY READING
	Library path: tuto_antares_legacy_models.yml
	Library loaded: antares_legacy_models
COMPONENTS DEFINITION
	Components defined: ['my_area', 'load']
INPUT SYSTEM CREATION
	The following input system has been created :
id='my_system' model_libraries='antares_legacy_models' components=[InputComponent(id='my_area', model='antares_legacy_models.area', scenario_group=None, parameters=[InputComponentParameter(id='spillage_cost', time_dependent=False, scenario_dependent=False, value=1000.0, scenario_group=None), InputComponentParameter(id='ens_cost', time_dependent=False, scenario_dependent=False, value=10000.0, scenario_group=None)]), InputComponent(id='load', model='antares_legacy_models.load', scenario_group=None, parameters=[InputComponentParameter(id='load', time_dependent=True, scenario_dependent=True, value='load', scenario_group=None)])] connections=None area_connections=None nodes=[]


### Step 2: Configure the Generator (1 unit of 100 MW)

Now we add a thermal generator to the area. This generator will have:
- 1 unit with a maximum capacity of 100 MW
- Generation cost of 50 $/MWh

In [ ]:
# Add a thermal generator component with technical and economic parameters
components.append(
    InputComponent(
        id="thermal_gen",
        model=f"{library_name}.thermal",
        parameters=[
            InputComponentParameter(id="p_min_unit", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_unit", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="generation_cost", time_dependent=False, scenario_dependent=False, value=50),
            InputComponentParameter(id="startup_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="fixed_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="d_min_up", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="d_min_down", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_min_cluster", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_cluster", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="nb_units_max", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="nb_units_min", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_forward", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_backward", time_dependent=False, scenario_dependent=False, value=0)
        ],
    )
)

print("\tComponents defined:", [comp.id for comp in components])

print("INPUT SYSTEM CREATION")
input_system = InputSystem(
            id = "my_system",
            model_libraries=library_name,
            components=components,
        )
print("\tThe following input system has been created :")
print(input_system)

	Components defined: ['my_area', 'load', 'thermal_gen']
INPUT SYSTEM CREATION
	The following input system has been created :
id='my_system' model_libraries='antares_legacy_models' components=[InputComponent(id='my_area', model='antares_legacy_models.area', scenario_group=None, parameters=[InputComponentParameter(id='spillage_cost', time_dependent=False, scenario_dependent=False, value=1000.0, scenario_group=None), InputComponentParameter(id='ens_cost', time_dependent=False, scenario_dependent=False, value=10000.0, scenario_group=None)]), InputComponent(id='load', model='antares_legacy_models.load', scenario_group=None, parameters=[InputComponentParameter(id='load', time_dependent=True, scenario_dependent=True, value='load', scenario_group=None)]), InputComponent(id='thermal_gen', model='antares_legacy_models.thermal', scenario_group=None, parameters=[InputComponentParameter(id='p_min_unit', time_dependent=False, scenario_dependent=False, value=0.0, scenario_group=None), InputComponentP

## Step 3 : Configure the port connection

Then, we interconnect the two components by configuring the ports connection

In [ ]:
from gems.study.parsing import InputPortConnections

# Initialize connections list
connections = []

# Add connections between the area and the thermal generator
connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="thermal_gen",
        port2="balance_port",
    )
)

# Add connection between the area and the load
connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="load",
        port2="balance_port",
    )
)


print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
            components=components,
            connections=connections,
        )
print("\tThe following input system has been created :")
print(input_system)



	Connections defined:
		my_area:balance_port <-> thermal_gen:balance_port
		my_area:balance_port <-> load:balance_port
	The following input system has been created :
id=None model_libraries=None components=[InputComponent(id='my_area', model='antares_legacy_models.area', scenario_group=None, parameters=[InputComponentParameter(id='spillage_cost', time_dependent=False, scenario_dependent=False, value=1000.0, scenario_group=None), InputComponentParameter(id='ens_cost', time_dependent=False, scenario_dependent=False, value=10000.0, scenario_group=None)]), InputComponent(id='load', model='antares_legacy_models.load', scenario_group=None, parameters=[InputComponentParameter(id='load', time_dependent=True, scenario_dependent=True, value='load', scenario_group=None)]), InputComponent(id='thermal_gen', model='antares_legacy_models.thermal', scenario_group=None, parameters=[InputComponentParameter(id='p_min_unit', time_dependent=False, scenario_dependent=False, value=0.0, scenario_group=None), 

## Step 4 : Run the study and Graph results

In [ ]:
from gems.simulation.optimization import (
    TimeBlock,
    build_problem,
)
from gems.study.resolve_components import (
    build_data_base,
    build_network,
    resolve_system
)
from gems.simulation.output_values import OutputValues

# Set optimization parameters
ScenarioNumber = 1 # first scenario
TimeSpan = 7*24 # one week with hourly time steps
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


# Printing results

print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

✓ Optimization problem solved successfully.
✓ Results extracted successfully.

 ===============  RESULTS  ===============
	Solver status: 0
	Objective value: 4403250.0

Component variables and outputs:

my_area : 
  spillage : [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

In these results, it's easy to see that the area has unsuplied energy while the load is above the 100MW threshold of thermal power generation.

In the next section, a wind farm and a solar farm are interconnected to the are for supply it enough energy.

# Interconnect Wind Farm and Solar Farm

## Step 1 : Creation of components 

In this section, the *wind_farm* and *solar_farm* components are added to our system

- The *wind_farm* has a fluctuated power generation between **0 and 60 MW**
- The *solar_farm* has a fluctuated power generation between **0 and 40 MW**


In [ ]:

# Add wind farm component with wind generation data from a data series
components.append(
    InputComponent(
        id="wind_farm",
        model=f"{library_name}.renewable",
        parameters=[
            InputComponentParameter(id="nominal_capacity", time_dependent=False, scenario_dependent=False, value=60),
            InputComponentParameter(id="unit_count", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="generation", time_dependent=True, scenario_dependent=True, value="wind"),
        ],              
    )
)

# Add connection between the area and the wind farm
connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="wind_farm",
        port2="balance_port",
    )
)

# Add solar farm component with solar generation data from a data series
components.append(
    InputComponent(
        id="solar_farm",
        model=f"{library_name}.renewable",
        parameters=[
            InputComponentParameter(id="nominal_capacity", time_dependent=False, scenario_dependent=False, value=40),
            InputComponentParameter(id="unit_count", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="generation", time_dependent=True, scenario_dependent=True, value="solar"),
        ],              
    )
)

# Add connection between the area and the solar farm
connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="solar_farm",
        port2="balance_port",
    )
)

print("\tComponents defined:", [comp.id for comp in components])
print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
    id = "my_system",
    components=components,
    connections=connections,
    )

print("\tThe following input system has been created :")
print(input_system)

	Components defined: ['my_area', 'load', 'thermal_gen', 'wind_farm', 'solar_farm']
	Connections defined:
		my_area:balance_port <-> thermal_gen:balance_port
		my_area:balance_port <-> load:balance_port
		my_area:balance_port <-> wind_farm:balance_port
		my_area:balance_port <-> solar_farm:balance_port
	The following input system has been created :
id='my_system' model_libraries=None components=[InputComponent(id='my_area', model='antares_legacy_models.area', scenario_group=None, parameters=[InputComponentParameter(id='spillage_cost', time_dependent=False, scenario_dependent=False, value=1000.0, scenario_group=None), InputComponentParameter(id='ens_cost', time_dependent=False, scenario_dependent=False, value=10000.0, scenario_group=None)]), InputComponent(id='load', model='antares_legacy_models.load', scenario_group=None, parameters=[InputComponentParameter(id='load', time_dependent=True, scenario_dependent=True, value='load', scenario_group=None)]), InputComponent(id='thermal_gen', mod

## Step 2 : Results visualization

In [7]:
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

✓ Optimization problem solved successfully.
✓ Results extracted successfully.

 ===============  RESULTS  ===============
	Solver status: 0
	Objective value: 834150.0

Component variables and outputs:

my_area : 
  spillage : [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

By adding the renewable generation to the system, there is no more unsupplied energy.

In the next section, the thermal generator will have several units, forcing the system to implement unit commitment.



# Add Unit Commitment

## Add Units in the thermal cluster

We configure the **10 units** inside the thermal generator cluster.

For each unit :
* the minimum power generated is set to 1 MW
* the maximum power generated is set to 10 MW

In [ ]:
# Delete the previous thermal generator
components = [comp for comp in components if comp.id != 'thermal_gen']

# Create a new thermal generator with different parameters
components.append(
    InputComponent(
        id="thermal_gen",
        model=f"{library_name}.thermal",
        parameters=[
            InputComponentParameter(id="p_min_unit", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_max_unit", time_dependent=False, scenario_dependent=False, value=10),
            InputComponentParameter(id="generation_cost", time_dependent=False, scenario_dependent=False, value=50),
            InputComponentParameter(id="startup_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="fixed_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="d_min_up", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="d_min_down", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_min_cluster", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_cluster", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="nb_units_max", time_dependent=False, scenario_dependent=False, value=10),
            InputComponentParameter(id="nb_units_min", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_forward", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_backward", time_dependent=False, scenario_dependent=False, value=0)
        ],
    )
)

# Add connections
connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="thermal_gen",
        port2="balance_port",
    )
)   

print("\tComponents defined:", [comp.id for comp in components])
print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
    id = "my_system",
    components=components,
    connections=connections,
    )   

	Components defined: ['my_area', 'load', 'wind_farm', 'solar_farm', 'thermal_gen']
	Connections defined:
		my_area:balance_port <-> thermal_gen:balance_port
		my_area:balance_port <-> load:balance_port
		my_area:balance_port <-> wind_farm:balance_port
		my_area:balance_port <-> solar_farm:balance_port
		my_area:balance_port <-> thermal_gen:balance_port


## Results Visualization

In [9]:
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

✓ Optimization problem solved successfully.
✓ Results extracted successfully.

 ===============  RESULTS  ===============
	Solver status: 0
	Objective value: 242950.0

Component variables and outputs:

my_area : 
  spillage : [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

The results illustrate how the number of thermal units generating power changes over the simulation week, reflecting the unit commitment feature. 

Indeed, the number of thermal units on fluctuated throughout the wind and solar power generation.